In [ ]:
# import sys
# import os
# sys.path.append(os.path.abspath(".."))

# os.environ["HOME"] = "/home/grzwia6937"

In [ ]:
import os
import re
from functools import reduce
from pathlib import Path

import pandas as pd
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

# use spark session
spark = (
    SparkSession.builder.master("local[*]")
    .appName("GenPM-fe")
    .config("spark.executor.memory", "24g")
    .config("spark.driver.memory", "16g")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.default.parallelism", "8")
    .config("spark.log.level", "ERROR")
    .getOrCreate()
)


eda_data_path = Path("/home/grzwia6937/app/apps/apps/generator/data/shared_dir/eda_data")
raw_pm_path = eda_data_path / "raw_pm_data"
pm_kpi_pivot = eda_data_path / "pm_data_pivot"
sample_path = eda_data_path / "sample"
pm_stats_path = eda_data_path / "stats"
pm_agg_path = eda_data_path / "agg"
pm_metadata = eda_data_path / "pm_metadata"

# Consts
MIN_KPI_COVERAGE = 0.9

PRESENTATION = False

In [ ]:
def fill_missing_timestamps(
    df: DataFrame,
    time_col: str,
    station_col: str,
) -> DataFrame:
    """
    Fill missing hourly timestamps per station using each station's
    own min/max time range. Operates in long format — safe for large data.
    """
    print("PREPROCESSING: FILLING MISSING TIMESTAMPS")
    # Per-station time bounds — small aggregation, stays distributed
    station_bounds = df.groupBy(station_col).agg(
        f.min(time_col).alias("min_t"), f.max(time_col).alias("max_t")
    )

    # Generate hourly spine per station using sequence + explode
    station_spines = station_bounds.withColumn(
        time_col, f.explode(f.sequence(f.col("min_t"), f.col("max_t"), f.expr("INTERVAL 1 HOUR")))
    ).drop("min_t", "max_t")

    # Left join original data — only fills gaps, no cross-station explosion
    return station_spines.join(df, on=[station_col, time_col], how="left")

In [ ]:
def prepare_pm_data(pm_df_long: DataFrame):
    print("PREPROCESSING PM DATA")
    # Simple df screening
    pm_df_long.printSchema()

    print(f"{pm_df_long.count()=}")

    # # pm_df entry cleaning
    # pm_df_long = pm_df_long.withColumnsRenamed({"bts_anon": "bts_id", "distname_anon": "distname"}
    # )
    pm_df_long = pm_df_long.dropDuplicates()
    pm_df_long = pm_df_long.dropna(subset=("start_time", "bts_id", "distname"))

    # Timestamp frequency uniformoty (1 hour) and KPI-bts recording range verification
    pm_df_long = fill_missing_timestamps(pm_df_long, "start_time", "bts_id")

    # # KPI version flattening
    # pm_df_long = _coalesce_kpi_version(pm_df_long, kpi_definitions)

    # sdm.write_parquet(pm_df_long, EDA_DATA_PATH / "debug_coalesced_kpi")
    # print("COALESCE SUCCESS")
    # at last pivot
    # pm_df_wide = _pivot_pm_data(pm_df_long)

    return pm_df_long


In [ ]:
raw_pm = spark.read.parquet(str(raw_pm_path))

In [ ]:
pm = prepare_pm_data(raw_pm)

In [ ]:
# pm.show(20)